# MVTec AD Data Exploration and Preprocessing

This notebook performs only reproducible MVTec AD inspection, validation, preprocessing, and reporting. Raw data is treated as read-only.

## 1. Environment Setup and Dataset Inspection

Configure paths once, inspect the actual directory structure, and retain a raw-file checksum manifest so later cells can prove the raw dataset was not changed.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import numpy as np
from PIL import Image, UnidentifiedImageError

RAW_DATA_DIR = Path('datasets/mvtec_ad')
PROCESSED_DATA_DIR = Path('datasets/mvtec_ad_processed')
IMAGE_SIZE = 256
SUPPORTED_IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
RGB_RESAMPLE = Image.Resampling.LANCZOS
MASK_RESAMPLE = Image.Resampling.NEAREST

assert RAW_DATA_DIR.is_dir(), f'Raw dataset directory not found: {RAW_DATA_DIR.resolve()}'
assert RAW_DATA_DIR.resolve() != PROCESSED_DATA_DIR.resolve(), 'Raw and processed paths must differ.'

def is_supported_image(path):
    return path.is_file() and path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS

def image_files(directory):
    return sorted(path for path in directory.rglob('*') if is_supported_image(path))

def category_directories(root=RAW_DATA_DIR):
    required = {'train', 'test', 'ground_truth'}
    return sorted(path for path in root.iterdir() if path.is_dir() and required.issubset({item.name for item in path.iterdir() if item.is_dir()}))

def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def dataset_manifest(root):
    return {str(path.relative_to(root)): (path.stat().st_size, file_sha256(path)) for path in sorted(root.rglob('*')) if path.is_file()}

CATEGORIES = category_directories()
assert CATEGORIES, f'No valid MVTec categories found under {RAW_DATA_DIR}'
RAW_FILE_MANIFEST = dataset_manifest(RAW_DATA_DIR)

print('MVTec AD Dataset')
print('=' * 50)
print(f'Raw dataset: {RAW_DATA_DIR.resolve()}')
print(f'Categories discovered: {len(CATEGORIES)}')
print('Categories:', ', '.join(category.name for category in CATEGORIES))
for category in CATEGORIES:
    print(f'\nCategory: {category.name}')
    for split in ('train', 'test', 'ground_truth'):
        split_dir = category / split
        class_counts = {subdir.name: len(image_files(subdir)) for subdir in sorted(split_dir.iterdir()) if subdir.is_dir()}
        print(f'  {split}/')
        for class_name, count in class_counts.items():
            print(f'    {class_name}: {count} image files')

root_non_directories = [path.name for path in RAW_DATA_DIR.iterdir() if not path.is_dir()]
print(f'\nRaw file manifest entries: {len(RAW_FILE_MANIFEST)}')
print('Unexpected top-level non-directory entries:', root_non_directories or 'none')

## 2. Dataset Statistics

Count every split and class discovered in the raw dataset, then measure decodable image properties without making assumptions about categories or anomaly labels.

In [ ]:
def split_records(split):
    records = []
    for category in CATEGORIES:
        split_dir = category / split
        for path in image_files(split_dir):
            relative = path.relative_to(split_dir)
            records.append({'category': category.name, 'split': split, 'label': relative.parts[0], 'path': path})
    return records

def probe_image(path):
    try:
        with Image.open(path) as image:
            image.load()
            return {'readable': True, 'size': image.size, 'mode': image.mode, 'channels': len(image.getbands()), 'format': image.format, 'error': None}
    except (UnidentifiedImageError, OSError, ValueError) as error:
        return {'readable': False, 'size': None, 'mode': None, 'channels': None, 'format': None, 'error': f'{type(error).__name__}: {error}'}

RAW_RECORDS = {split: split_records(split) for split in ('train', 'test', 'ground_truth')}
IMAGE_STATISTICS = {}
for split, records in RAW_RECORDS.items():
    probes = [probe_image(record['path']) for record in records]
    readable = [probe for probe in probes if probe['readable']]
    IMAGE_STATISTICS[split] = {
        'total': len(records), 'readable': len(readable),
        'formats': Counter(probe['format'] for probe in readable),
        'modes': Counter(probe['mode'] for probe in readable),
        'channels': Counter(probe['channels'] for probe in readable),
        'dimensions': Counter(probe['size'] for probe in readable),
    }

print('Dataset Statistics')
print('=' * 50)
print(f'Categories: {len(CATEGORIES)}')
print(f"Total training images: {len(RAW_RECORDS['train'])}")
print(f"Total test images: {len(RAW_RECORDS['test'])}")
print(f"Total ground-truth masks: {len(RAW_RECORDS['ground_truth'])}")
print('\nPer-category counts')
print(f"{'Category':<14} {'Train':>7} {'Test':>7} {'Good test':>10} {'Anomalous':>10} {'Masks':>7}")
for category in CATEGORIES:
    name = category.name
    train_count = sum(record['category'] == name for record in RAW_RECORDS['train'])
    test_records = [record for record in RAW_RECORDS['test'] if record['category'] == name]
    good_count = sum(record['label'] == 'good' for record in test_records)
    mask_count = sum(record['category'] == name for record in RAW_RECORDS['ground_truth'])
    print(f'{name:<14} {train_count:>7} {len(test_records):>7} {good_count:>10} {len(test_records) - good_count:>10} {mask_count:>7}')
print('\nImage properties by split')
for split, stats in IMAGE_STATISTICS.items():
    dimensions = stats['dimensions']
    widths = [size[0] for size in dimensions for _ in range(dimensions[size])]
    heights = [size[1] for size in dimensions for _ in range(dimensions[size])]
    print(f"{split}: readable={stats['readable']}/{stats['total']}, formats={dict(stats['formats'])}, modes={dict(stats['modes'])}, channels={dict(stats['channels'])}")
    print(f"  dimensions: min={(min(widths), min(heights))}, max={(max(widths), max(heights))}, unique={len(dimensions)}")

## 3. Train Image Validation

Validate every raw training image without changing it, and report unsupported or unreadable files rather than deleting them.

In [ ]:
def unexpected_files(split):
    files = []
    for category in CATEGORIES:
        split_dir = category / split
        files.extend(path for path in split_dir.rglob('*') if path.is_file() and not is_supported_image(path))
    return sorted(files)

def validate_raw_images(split):
    records, failures = RAW_RECORDS[split], []
    results = []
    for record in records:
        path = record['path']
        result = {**record, **probe_image(path), 'exists': path.exists(), 'supported_extension': path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS}
        results.append(result)
        if not result['exists'] or not result['supported_extension'] or not result['readable']:
            failures.append(result)
    return {'split': split, 'records': results, 'failures': failures, 'unexpected': unexpected_files(split)}

TRAIN_VALIDATION = validate_raw_images('train')
train_modes = Counter(record['mode'] for record in TRAIN_VALIDATION['records'] if record['readable'])
train_dimensions = Counter(record['size'] for record in TRAIN_VALIDATION['records'] if record['readable'])
print('Train Image Validation')
print('=' * 50)
print(f"Total images: {len(TRAIN_VALIDATION['records'])}")
print(f"Readable images: {sum(record['readable'] for record in TRAIN_VALIDATION['records'])}")
print(f"Corrupted or invalid images: {len(TRAIN_VALIDATION['failures'])}")
print(f"Image modes: {dict(train_modes)}")
print(f"Unique dimensions: {len(train_dimensions)}")
print(f"Unexpected files under train/: {len(TRAIN_VALIDATION['unexpected'])}")
if TRAIN_VALIDATION['failures']:
    print('Corrupted files:')
    for record in TRAIN_VALIDATION['failures']:
        print(f"- {record['path']}: {record['error'] or 'invalid file'}")
if TRAIN_VALIDATION['unexpected']:
    print('Unexpected files:')
    for path in TRAIN_VALIDATION['unexpected']:
        print(f'- {path}')
assert not TRAIN_VALIDATION['failures'], 'Raw training validation failed; raw files were not modified.'